In [1]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    temperature = 0,
    groq_api_key = "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",
    model_name = "llama-3.3-70b-versatile"
)
result = llm.invoke("what is an apple")
print(result.content)

A simple yet delicious question!

An apple is a type of fruit that grows on apple trees (Malus domestica). It is a juicy, sweet, and crunchy fruit that is one of the most widely consumed fruits in the world.

Apples are typically round or oval in shape, with a thin skin that can range in color from red, green, yellow, to sometimes a combination of these colors. The flesh of the apple is firm and juicy, with a central core containing seeds.

Apples are a great source of nutrients, including:

1. Fiber: Apples are high in dietary fiber, which can help promote digestive health and support healthy blood sugar levels.
2. Antioxidants: Apples contain a range of antioxidants, including quercetin and catechins, which can help protect against cell damage and reduce the risk of chronic diseases.
3. Vitamins and minerals: Apples are a good source of vitamins A and C, as well as minerals like potassium and manganese.

Apples come in over 7,500 known varieties, each with its own unique characterist

In [16]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from langchain.embeddings import HuggingFaceBgeEmbeddings
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os

app = Flask(__name__)
CORS(app)

# Initialize LLM
def initialize_llm():
    from langchain_groq import ChatGroq
    llm = ChatGroq(
        temperature=0,
        groq_api_key="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",
        model_name="llama-3.3-70b-versatile"
    )
    return llm


# Load or Create Vector DB
def load_or_create_vector_db():
    db_path = "C:/arya/4 TH YEAR/MAJOR PROJECT/Corner/backend/src/content"
    persist_directory = "./chroma_db"
    
    if not os.path.exists(persist_directory):
        loader = DirectoryLoader(db_path, glob="*.pdf", loader_cls=PyPDFLoader)
        documents = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        texts = text_splitter.split_documents(documents)
        embeddings = HuggingFaceBgeEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
        vector_db = Chroma.from_documents(texts, embeddings, persist_directory=persist_directory)
        vector_db.persist()
    else:
        embeddings = HuggingFaceBgeEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
        vector_db = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
        
    return vector_db


# Setup the chatbot pipeline
def setup_qa_chain(vector_db, llm):
    retriever = vector_db.as_retriever()
    prompt_template = """
    You are a compassionate mental health chatbot. Respond thoughtfully to the following question:
    {context}
    User: {question}
    Chatbot: """
    
    PROMPT = PromptTemplate(template=prompt_template, input_variables=['context', 'question'])
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        chain_type_kwargs={"prompt": PROMPT}
    )
    return qa_chain


# Initialize components
llm = initialize_llm()
vector_db = load_or_create_vector_db()
qa_chain = setup_qa_chain(vector_db, llm)

# API route to handle chatbot queries
@app.route("/chatbot", methods=["POST"])
def chatbot_response():
    query = request.json.get("query")
    if not query:
        return jsonify({"response": "Please provide a query"}), 400

    response = qa_chain.run(query)
    return jsonify({"response": response})


if __name__ == "__main__":
    app.run(port=5001)


Intializing Chatbot.........


C:\Users\Arya\AppData\Local\Temp\ipykernel_9164\3491828963.py:24: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_db.persist()


ChromaDB created and data saved


C:\Users\Arya\AppData\Local\Temp\ipykernel_9164\3491828963.py:65: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa_chain.run(query)


Chatbot: Hello! It's lovely to connect with you. I see you've been reading about mental health, specifically about how it's defined and the importance of considering the social and physical environments that influence our well-being. It's fascinating to explore the different perspectives on what constitutes a good life and how mental health is intertwined with our relationships, self-respect, and sense of purpose.

The World Health Organization's definition of positive mental health is a great starting point, but as you've noted, it's essential to consider the broader context in which we live. Our mental health is indeed influenced by our interactions with others and our environment, and it's crucial to recognize the impact of these factors on our overall well-being.

I'm here to listen and support you in any way I can. What are your thoughts on mental health, and how do you think we can promote positive mental health in our daily lives? Is there something specific that's been on your 

KeyboardInterrupt: Interrupted by user